# QUERY 3
Cuenta de origen y monto de transacciones USD en el período [2022-09-06, 2022-09-15]
con monto menor a 1 centésimo del promedio encontrado para el mismo formato de
pago en el período [2022-09-01, 2022-09-05]

In [1]:
import pandas as pd

RUTA_DE_DATASETS             = "../../datasets"
# NOMBRE_DATASET      = "trans_sample.csv"
NOMBRE_DATASET      = "transacciones_sample_q4.csv"
RUTA_RESULTADO_QUERY3        = "q3_solucion.csv"

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_DATASET}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)

guardar_csv = lambda df, ruta: df.to_csv(ruta, index=False)
# Filtrar USD antes de samplear
transacciones = transacciones_completas[
    transacciones_completas["Payment Currency"] == "US Dollar"
]

In [2]:
transacciones["Amount Paid"] = pd.to_numeric(transacciones["Amount Paid"], errors="coerce")
transacciones = transacciones.dropna(subset=["Amount Paid", "Payment Format"])

periodo_temprano = transacciones[
    (transacciones["Timestamp"] >= "2022/09/01") &
    (transacciones["Timestamp"] <= "2022/09/06")
]
periodo_tardio = transacciones[
    (transacciones["Timestamp"] >= "2022/09/06") &
    (transacciones["Timestamp"] <= "2022/09/15")
]

print(f"Período temprano (1/9-5/9): {len(periodo_temprano)} transacciones")
print(f"Período tardío  (6/9-15/9): {len(periodo_tardio)} transacciones")

Período temprano (1/9-5/9): 1393082 transacciones
Período tardío  (6/9-15/9): 1160806 transacciones


In [3]:
import hashlib

def obtener_id_shard(valor, total_shards):
    valor_norm = str(valor).strip()
    hash_hex = hashlib.md5(valor_norm.encode('utf-8')).hexdigest()
    return (int(hash_hex, 16) % total_shards) + 1

formatos = ["ACH", "Wire", "Cheque", "Cash", "Credit Card", "Reinvestment"]
for f in formatos:
    print(f"{f} -> shard {obtener_id_shard(f, 3)}")

ACH -> shard 3
Wire -> shard 2
Cheque -> shard 3
Cash -> shard 3
Credit Card -> shard 2
Reinvestment -> shard 3


In [4]:
stats_por_formato = (
    periodo_temprano
    .groupby("Payment Format")["Amount Paid"]
    .agg(suma="sum", count="count")
)
stats_por_formato["promedio"] = stats_por_formato["suma"] / stats_por_formato["count"]

print(stats_por_formato)

                        suma   count      promedio
Payment Format                                    
ACH             2.288207e+11  161619  1.415803e+06
Bitcoin         1.964827e+04       9  2.183141e+03
Cash            8.483899e+10  124340  6.823146e+05
Cheque          2.440185e+11  472553  5.163833e+05
Credit Card     1.161857e+09  336672  3.451007e+03
Reinvestment    6.380787e+10  253974  2.512378e+05
Wire            1.976411e+10   43915  4.500537e+05


In [5]:
df = periodo_tardio.copy()

df["Promedio Formato"] = df["Payment Format"].map(stats_por_formato["promedio"] )
df = df.dropna(subset=["Promedio Formato"])

resultado_query3 = df[
    df["Amount Paid"] < df["Promedio Formato"] * 0.01
 ][["From Bank", "Account", "Payment Format", "Amount Paid"]].reset_index(drop=True)

guardar_csv(resultado_query3, RUTA_RESULTADO_QUERY3)
print(f"Resultados Q3: {len(resultado_query3)} transacciones")
resultado_query3.head()

Resultados Q3: 663648 transacciones


,From Bank,Account,Payment Format,Amount Paid
0,011495,8008AD900,ACH,10149.61
1,023,800077210,ACH,2680.03
2,023,800077210,ACH,10993.76
3,023,800077210,ACH,5845.00
4,023,800077210,ACH,8534.40


In [6]:
# Validación del resultado
df_validacion = resultado_query3.copy()
df_validacion = df_validacion.merge(
    periodo_tardio[["Account", "Payment Format", "Timestamp", "Amount Paid"]],
    on=["Account", "Payment Format", "Amount Paid"],
    how="left"
)
df_validacion["Promedio Formato"] = df_validacion["Payment Format"].map(stats_por_formato["promedio"])
df_validacion["1% Promedio"] = df_validacion["Promedio Formato"] * 0.01
df_validacion["Es valido"] = df_validacion["Amount Paid"] < df_validacion["1% Promedio"]
df_validacion

,From Bank,Account,Payment Format,Amount Paid,Timestamp,Promedio Formato,1% Promedio,Es valido
0,011495,8008AD900,ACH,10149.61,2022/09/06 03:39,1.415803e+06,14158.033978,True
1,023,800077210,ACH,2680.03,2022/09/06 06:47,1.415803e+06,14158.033978,True
2,023,800077210,ACH,10993.76,2022/09/08 12:09,1.415803e+06,14158.033978,True
3,023,800077210,ACH,5845.00,2022/09/09 02:47,1.415803e+06,14158.033978,True
4,023,800077210,ACH,8534.40,2022/09/09 05:20,1.415803e+06,14158.033978,True
...,...,...,...,...,...,...,...,...
2658759,013474,803A93631,ACH,0.23,2022/09/10 23:31,1.415803e+06,14158.033978,True
2658760,013474,803A93631,ACH,0.01,2022/09/06 05:52,1.415803e+06,14158.033978,True
2658761,013474,803A93631,ACH,0.01,2022/09/08 18:41,1.415803e+06,14158.033978,True
2658762,013474,803A93631,ACH,0.01,2022/09/09 13:10,1.415803e+06,14158.033978,True


In [7]:
print(f"Período temprano total: {len(periodo_temprano)}")
print(f"Período tardío total: {len(periodo_tardio)}")
print("\nTemprano por formato:")
print(periodo_temprano.groupby("Payment Format")["Amount Paid"].count())
print("\nTardío por formato:")
print(periodo_tardio.groupby("Payment Format")["Amount Paid"].count())

Período temprano total: 1393082
Período tardío total: 1160806

Temprano por formato:
Payment Format
ACH             161619
Bitcoin              9
Cash            124340
Cheque          472553
Credit Card     336672
Reinvestment    253974
Wire             43915
Name: Amount Paid, dtype: int64

Tardío por formato:
Payment Format
ACH            156710
Bitcoin             9
Cash           125739
Cheque         485180
Credit Card    349909
Wire            43259
Name: Amount Paid, dtype: int64
